# Etapa 4 — Aprendizado Probabilístico: MLE, MAP e Algoritmo Expectation-Maximization (EM)

**Pergunta prática:** Dado o perfil e os sintomas de um paciente com dengue, como estimar os parâmetros probabilísticos (MLE e MAP) e descobrir componentes latentes não observadas via algoritmo EM para a predição de hospitalização?

**Dados:** Sinan/Dengue 2026 (`DENGBR26.csv`), Portal de Dados Abertos do SUS.

Rode as células em ordem (Ambiente de execução → Executar tudo).

In [ ]:
import argparse
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from em import (LatentClassNB, NaiveBayes, bernoulli_curves, bernoulli_map, bernoulli_mle)
from utils import (C_AQUA, C_BLUE, C_MUTED, C_ORANGE, FEATURES_NB, STATE_NAMES, balanced_indices,
                   check_monotonic, classification_report, ensure_out_dir, load_dengue,
                   prepare_nb_dataset, save_csv, save_json, set_seed, setup_matplotlib,
                   stratified_kfold, stratified_split, OUT_DIR)

ALPHA = 2.0          # Prior Dirichlet(2) = Beta(2,2) nas variáveis binárias (Suavização de Laplace)
METRICS = ["acuracia", "acuracia_balanceada", "recall", "precisao", "f1", "auc", "log_loss"]
CAMINHO_CSV = "/content/DENGBR26.csv"

## 1. Funções Auxiliares e Métricas
Esta seção estabelece o suporte computacional, a avaliação de verossimilhança e os critérios estatísticos de comparação entre os modelos Naive Bayes puro e Naive Bayes estendido por variáveis latentes (EM).


In [ ]:
def md_table(df: pd.DataFrame) -> str:
    """Converte um DataFrame do pandas em uma tabela formatada em Markdown."""
    cols = list(df.columns)
    esc = lambda v: str(v).replace("|", "\\|")
    out = ["| " + " | ".join(esc(c) for c in cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(esc(v) for v in r.values) + " |")
    return "\n".join(out)


def joint_ll(model, X, y):
    """Calcula a log-verossimilhança conjunta log P(x, c) somada sobre as amostras."""
    lj = model.log_joint(X)
    return float(lj[np.arange(len(y)), np.searchsorted(model.classes_, y)].sum())


def paired(res, key):
    """Realiza comparação pareada das métricas entre o NB puro e as variantes com EM."""
    nb = np.array([d[key] for d in res["NB puro"]])
    out = {}
    for nm in ["NB + EM (K=2)", "NB + EM (K=3)"]:
        d = np.array([x[key] for x in res[nm]]) - nb
        out[nm] = (d.mean(), int(np.sum(d > 0)), len(d))
    return out


def verdict(p, key="auc", thr=0.002):
    """Gera veredito textual sobre o ganho de desempenho das variantes do EM."""
    parts = []
    for nm, (d, wins, n) in p.items():
        tag = "ganho consistente" if (d > thr and wins == n) else ("ganho marginal" if d > 0 else "sem ganho")
        parts.append(f"{nm}: Δ{key} = {d:+.4f} (melhor em {wins}/{n} folds → {tag})")
    return "; ".join(parts)

## 2. Comparação dos Estimadores MLE e MAP (Bernoulli)
Esta seção investiga as diferenças teóricas e práticas entre estimar probabilidades por Frequência Relativa/Máxima Verossimilhança (MLE) e por Máximo a Posteriori (MAP) utilizando uma prior Beta/Dirichlet ($\alpha = 2.0$, equivalente à Suavização de Laplace).

In [ ]:
def stage_mle_map(y, plt, alvo):
    """Compara os estimadores de Máxima Verossimilhança (MLE) e Máximo a Posteriori (MAP) sob diferentes tamanhos de amostra."""
    set_seed()
    theta_full = bernoulli_mle(y)
    rows = []
    for n in [5, 10, 20, 50, 100, 500, 2000]:
        e_mle, e_map = [], []
        for _ in range(300):
            s = y[np.random.choice(len(y), n, replace=False)]
            e_mle.append(abs(bernoulli_mle(s) - theta_full))
            e_map.append(abs(bernoulli_map(s, 2, 2) - theta_full))
        rows.append((n, np.mean(e_mle), np.mean(e_map)))
    err = pd.DataFrame(rows, columns=["n", "erro_MLE", "erro_MAP"])

    # Amostra pedagógica: n=20 com k=1 sucesso
    while True:
        s = y[np.random.choice(len(y), 20, replace=False)]
        if s.sum() == 1:
            break
    k, n = int(s.sum()), len(s)
    th_mle, th_map = bernoulli_mle(s), bernoulli_map(s, 2, 2)
    g, lik, prior, post = bernoulli_curves(k, n, 2, 2)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(g, lik, color=C_BLUE, label="verossimilhança")
    ax[0].plot(g, prior, color=C_MUTED, ls="--", label="prior Beta(2,2)")
    ax[0].plot(g, post, color=C_ORANGE, label="posteriori")
    ax[0].axvline(th_mle, color=C_BLUE, lw=1.2, alpha=.7, label=f"MLE = {th_mle:.3f}")
    ax[0].axvline(th_map, color=C_ORANGE, lw=1.2, alpha=.7, label=f"MAP = {th_map:.3f}")
    ax[0].axvline(theta_full, color="k", lw=1, ls=":", label=f"θ (base toda) = {theta_full:.3f}")
    ymax = max(lik.max(), post.max()); ax[0].set_ylim(0, ymax * 1.1); ax[0].set_xlim(0, .5)
    ax[0].set_title(f"MLE vs MAP em amostra pequena (n={n}, k={k})")
    ax[0].set_xlabel(f"θ = P({alvo} = sim)"); ax[0].set_ylabel("densidade (normalizada)")
    ax[0].legend(loc="upper right", fontsize=9)

    ax[1].plot(err.n, err.erro_MLE, "o-", color=C_BLUE, label="MLE")
    ax[1].plot(err.n, err.erro_MAP, "o-", color=C_ORANGE, label="MAP Beta(2,2)")
    ax[1].set_xscale("log"); ax[1].set_xlabel("tamanho da amostra n")
    ax[1].set_ylabel("erro absoluto médio |θ̂ − θ| (300 réplicas)")
    ax[1].set_title("MAP e MLE convergem quando n cresce"); ax[1].legend()
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "etapa4_mle_map.png"), dpi=160); plt.close(fig)
    return theta_full, err, dict(k=k, n=n, mle=th_mle, map=th_map)

## 3. Avaliação Hold-out 70/30: Naive Bayes e EM (Monotonicidade da Log-Verossimilhança)
Esta seção realiza o ajuste de modelos em um único split estratificado de treino (70%) e teste (30%), focando na análise de convergência e na monotonicidade do algoritmo Expectation-Maximization.

In [ ]:
def stage_holdout(X, y, cards, features, plt, a):
    """Ajusta os modelos Naive Bayes e EM (K=2, K=3) em hold-out 70/30 e avalia a verossimilhança."""
    set_seed()
    tr, te = stratified_split(y, 0.3)
    Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]

    nb = NaiveBayes(ALPHA).fit(Xtr, ytr, cards)
    models = {"NB": nb}
    for K in (2, 3):
        set_seed()
        models[f"EM_K{K}"] = LatentClassNB(K, ALPHA, a.max_iter, a.tol, a.n_init).fit(Xtr, ytr, cards)
    mle = {}
    for K in (2, 3):
        set_seed()
        mle[K] = LatentClassNB(K, 1.0, a.max_iter, a.tol, a.n_init).fit(Xtr, ytr, cards)

    ll_nb_tr = nb.log_likelihood(Xtr, ytr)
    info = {}
    for K in (2, 3):
        h = np.array(mle[K].ll_history_)
        n99 = int(np.argmax(h - h[0] >= .99 * (h[-1] - h[0]))) + 1
        info[K] = dict(iters=mle[K].n_iter_, iters_99=n99, monotonic=mle[K].monotonic_,
                       worst=mle[K].worst_drop_, ll0=h[0], llf=h[-1],
                       map_mono=models[f"EM_K{K}"].monotonic_, map_iters=models[f"EM_K{K}"].n_iter_)

    # Figura: log-verossimilhança por iteração
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for K, col in [(2, C_BLUE), (3, C_ORANGE)]:
        h = np.array(mle[K].ll_history_)
        ax[0].plot(np.arange(1, len(h) + 1), h, color=col, label=f"EM, Z com {K} componentes")
        ax[1].plot(np.arange(2, len(h) + 1), np.maximum(np.diff(h), 1e-12), color=col, label=f"K = {K}")
        ax[0].axvline(info[K]["iters_99"], color=col, lw=1, ls=":")
        ax[0].text(info[K]["iters_99"] + 1, h[0] + (h[-1] - h[0]) * (0.18 if K == 2 else 0.06),
                   f"99% do ganho: it. {info[K]['iters_99']}", color=col, fontsize=8)
    ax[0].axhline(ll_nb_tr, color=C_MUTED, ls="--", lw=1.2)
    ax[0].text(len(mle[3].ll_history_) * .55, ll_nb_tr - 2500, "Naive Bayes puro (K = 1)", color=C_MUTED, fontsize=9, va="top")
    from matplotlib.ticker import FuncFormatter
    ax[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1e6:.2f}"))
    ax[0].set_xlabel("iteração do EM"); ax[0].set_ylabel("log P(x, c) no treino (em milhões)")
    ax[0].set_title("Log-verossimilhança cresce a cada iteração"); ax[0].legend(loc="lower right")
    ax[1].set_yscale("log"); ax[1].set_xlabel("iteração do EM")
    ax[1].set_ylabel("ganho  L(t) − L(t−1)  (escala log)")
    ax[1].set_title("Todos os ganhos são positivos (monotonicidade)"); ax[1].legend()
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "etapa4_em_loglik.png"), dpi=160); plt.close(fig)

    # Avaliação no conjunto de teste
    proba = {m: mdl.predict_proba(Xte)[:, 1] for m, mdl in models.items()}
    rep = {m: classification_report(yte, p) for m, p in proba.items()}
    ll_te = {m: joint_ll(mdl, Xte, yte) / len(yte) for m, mdl in models.items()}
    ll_tr = {m: joint_ll(mdl, Xtr, ytr) / len(ytr) for m, mdl in models.items()}

    # Exportação de predições e parâmetros para etapas subsequentes
    dfp = pd.DataFrame(Xte, columns=features)
    dfp.insert(0, "y_true", yte)
    dfp["p_nb"], dfp["p_em_k2"], dfp["p_em_k3"] = proba["NB"], proba["EM_K2"], proba["EM_K3"]
    save_csv(dfp, "etapa4_probabilidades_teste.csv")
    for K in (2, 3):
        save_csv(models[f"EM_K{K}"].parameters_table(features, STATE_NAMES), f"etapa4_componentes_z_k{K}.csv")
    return dict(models=models, mle=mle, info=info, rep=rep, ll_te=ll_te, ll_tr=ll_tr, n_tr=len(ytr),
                n_te=len(yte), ll_nb_tr=ll_nb_tr, Xtr=Xtr, ytr=ytr)

## 4. Validação Cruzada 5-Fold (Cenários Natural e Balanceado)
Esta seção avalia a capacidade preditiva discriminativa dos modelos via validação cruzada 5-fold estratificada, comparando o desequilíbrio original das notificações com um cenário balanceado artificialmente.

In [ ]:
def cv_compare(X, y, cards, a):
    """Executa validação cruzada 5-fold comparando Baseline, NB puro e NB + EM."""
    set_seed()
    folds = stratified_kfold(y, a.folds)
    names = ["Baseline (maioria)", "NB puro", "NB + EM (K=2)", "NB + EM (K=3)"]
    res = {m: [] for m in names}
    for i, (tr, te) in enumerate(folds):
        prev = np.full(len(te), y[tr].mean())
        res[names[0]].append(classification_report(y[te], prev))
        nb = NaiveBayes(ALPHA).fit(X[tr], y[tr], cards)
        res[names[1]].append(classification_report(y[te], nb.predict_proba(X[te])[:, 1]))
        for K, nm in [(2, names[2]), (3, names[3])]:
            em = LatentClassNB(K, ALPHA, a.max_iter, a.tol, a.n_init).fit(X[tr], y[tr], cards)
            res[nm].append(classification_report(y[te], em.predict_proba(X[te])[:, 1]))
        print(f"    fold {i + 1}/{a.folds} ok", flush=True)
    rows = []
    for m in names:
        r = {"modelo": m}
        for k in METRICS:
            v = np.array([d[k] for d in res[m]])
            r[k] = f"{v.mean():.4f} ± {v.std():.4f}"
        rows.append(r)
    return pd.DataFrame(rows), res


def plot_comparison(cv_nat, cv_bal, plt):
    """Gera gráficos comparativos de desempenho nos cenários natural e balanceado."""
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    colors = [C_MUTED, C_BLUE, C_ORANGE, C_AQUA]
    show = [("acuracia", "acurácia"), ("acuracia_balanceada", "acur. balanceada"), ("recall", "recall"), ("auc", "AUC")]
    for ax, (title, res) in zip(axs, [("Prevalência natural (≈ 8,6% internados)", cv_nat),
                                      ("Base balanceada 1:1 (subamostragem)", cv_bal)]):
        w = 0.2
        for i, (m, col) in enumerate(zip(res.keys(), colors)):
            vals = [np.mean([d[k] for d in res[m]]) for k, _ in show]
            xs = np.arange(len(show)) + (i - 1.5) * w
            ax.bar(xs, vals, width=w - .02, color=col, label=m)
            for x, v in zip(xs, vals):
                ax.text(x, v + .008, f"{v:.2f}", ha="center", fontsize=7, color=C_MUTED)
        ax.set_xticks(np.arange(len(show))); ax.set_xticklabels([s for _, s in show])
        ax.set_ylim(0, 1.08); ax.set_title(title); ax.grid(axis="x", visible=False)
    axs[0].set_ylabel("média nos folds (5-fold estratificado)")
    axs[0].legend(loc="upper center", ncol=2, fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "etapa4_comparacao.png"), dpi=160); plt.close(fig)

## 5. Execução do Pipeline e Exportação de Resultados
Esta parte corresponde ao bloco final do script/notebook, encarregado de ler os parâmetros da execução, chamar as etapas em ordem e compilar os artefatos finais.

In [ ]:
ap = argparse.ArgumentParser()
ap.add_argument("--csv", default=None)
ap.add_argument("--alvo", default="INTERNACAO", choices=["INTERNACAO", "GRAVE"])
ap.add_argument("--folds", type=int, default=5)
ap.add_argument("--nrows", type=int, default=None)
ap.add_argument("--max_iter", type=int, default=300)
ap.add_argument("--tol", type=float, default=1e-5)
ap.add_argument("--n_init", type=int, default=3)
a, _ = ap.parse_known_args()

here = os.getcwd()
cand = [a.csv] if a.csv else [os.path.join(here, p) for p in ("DENGBR26.csv", "DENGBR26.csv.zip", "data/DENGBR26.csv", "data/DENGBR26.csv.zip")]
csv = next((c for c in cand if c and os.path.exists(c)), None)
if csv is None:
    sys.exit("CSV não encontrado: coloque o DENGBR26.csv na pasta ou especifique via --csv.")

set_seed(); ensure_out_dir(); plt = setup_matplotlib()
print(f"[1/5] Lendo {csv} ...")
df = load_dengue(csv, nrows=a.nrows)
X, y, features, cards = prepare_nb_dataset(df, a.alvo)
print(f"      amostras válidas = {len(y):,} | prevalência({a.alvo}=sim) = {y.mean():.4f} | atributos = {features}")

print("[2/5] MLE e MAP (Bernoulli) ...")
theta, err, ex = stage_mle_map(y, plt, a.alvo)

print("[3/5] Hold-out 70/30: Naive Bayes + EM (log-verossimilhança) ...")
h = stage_holdout(X, y, cards, features, plt, a)
for K in (2, 3):
    i = h["info"][K]
    print(f"      K={K}: {i['iters']} it. (99% do ganho em {i['iters_99']}) | monotônica(MLE)={i['monotonic']} | monotônica(MAP, objetivo)={i['map_mono']}")

print("[4/5] Validação cruzada — cenário natural ...")
tab_nat, cv_nat = cv_compare(X, y, cards, a)

print("[5/5] Validação cruzada — cenário balanceado ...")
set_seed(); bi = balanced_indices(y)
tab_bal, cv_bal = cv_compare(X[bi], y[bi], cards, a)
plot_comparison(cv_nat, cv_bal, plt)

# Exportação final dos parâmetros e relatórios
p_nat, p_bal = paired(cv_nat, "auc"), paired(cv_bal, "acuracia")
p_bal_auc = paired(cv_bal, "auc")
p_nat_ll = paired(cv_nat, "log_loss")
comp = h["models"]["EM_K2"]
prof = []
for ci in range(2):
    for k in range(2):
        t = comp.theta_[ci][k]; off = comp.offsets_
        row = {"classe": a.alvo + (" = sim" if ci else " = não"), "z": k, "P(z|classe)": f"{comp.pi_[ci][k]:.3f}"}
        for f in ["COMORB", "VOMITO", "NAUSEA", "PETEQUIA", "LEUCOPENIA"]:
            j = features.index(f); row[f"P({f}=sim)"] = f"{t[off[j] + 1]:.3f}"
        j = features.index("FAIXA5"); row["P(60+)"] = f"{t[off[j] + 4]:.3f}"
        prof.append(row)
prof = pd.DataFrame(prof)

ll_rows = pd.DataFrame([{"modelo": m, "logL/amostra treino": f"{h['ll_tr'][m]:.4f}", "logL/amostra teste": f"{h['ll_te'][m]:.4f}"} for m in h["models"]])
conv = pd.DataFrame([{"K": K, "iterações (MLE)": h["info"][K]["iters"], "it. p/ 99% do ganho": h["info"][K]["iters_99"], "L inicial": f"{h['info'][K]['ll0']:,.1f}", "L final": f"{h['info'][K]['llf']:,.1f}", "monotônica": h["info"][K]["monotonic"], "menor ganho/it. (>0 ⇒ monotônica)": f"{h['info'][K]['worst']:.2e}"} for K in (2, 3)])

md = f"""# Etapa 4 — Aprendizado Probabilístico e EM (SUS / Dengue 2026)

Alvo: **{a.alvo} = sim** | amostras válidas: {len(y):,} | prevalência: {y.mean():.4f}
Atributos ({len(features)}): {', '.join(features)} | seed = 42 | prior Dirichlet(α={ALPHA}) ≡ Beta(2,2) nas binárias

## 1–2. MLE e MAP (Bernoulli)
θ̂_MLE (base toda) = **{theta:.4f}**. Exemplo pedagógico (n={ex['n']}, k={ex['k']}): MLE = {ex['mle']:.4f}, MAP Beta(2,2) = {ex['map']:.4f}.
Erro absoluto médio vs. θ da base (300 réplicas):

{md_table(err.round(5).assign(n=err.n.astype(int).astype(str)))}

## 3–5. Convergência do EM (hold-out 70/30, treino n = {h['n_tr']:,})
{md_table(conv)}

Log-verossimilhança média por amostra (log P(x, c)):

{md_table(ll_rows)}

## 6. Naive Bayes puro vs. Naive Bayes + EM — {a.folds}-fold estratificado (média ± desvio)

### Cenário A — prevalência natural (limiar 0,5)
{md_table(tab_nat)}

### Cenário B — base balanceada 1:1 (subamostragem da classe majoritária)
{md_table(tab_bal)}

Comparação pareada contra o NB puro:
- A (AUC): {verdict(p_nat)}
- A (log-loss; menor é melhor): {'; '.join(f'{k}: Δ = {v[0]:+.4f}' for k, v in p_nat_ll.items())}
- B (acurácia): {verdict(p_bal, 'acuracia')}
- B (AUC): {verdict(p_bal_auc)}

## Perfil dos componentes de Z (K = 2, ajustado no treino)
{md_table(prof)}
"""

with open(os.path.join(OUT_DIR, "etapa4_resultados.md"), "w", encoding="utf-8") as f:
    f.write(md)

save_json({
    "alvo": a.alvo, "features": features, "cards": cards, "state_names": {f: STATE_NAMES[f] for f in features},
    "theta_mle": theta, "alpha": ALPHA,
    "naive_bayes": {"prior": h["models"]["NB"].prior_, "likelihood": {f: {STATE_NAMES[f][v]: h["models"]["NB"].theta_[j][:, v] for v in range(cards[j])} for j, f in enumerate(features)}},
    "em": {f"K{K}": {"pi_z_dado_classe": h["models"][f"EM_K{K}"].pi_, "iteracoes": h["info"][K]["iters"], "loglik_historico": h["mle"][K].ll_history_} for K in (2, 3)},
    "metricas_holdout": h["rep"],
    "metricas_cv_natural": {m: {k: float(np.mean([d[k] for d in v])) for k in METRICS} for m, v in cv_nat.items()},
    "metricas_cv_balanceado": {m: {k: float(np.mean([d[k] for d in v])) for k in METRICS} for m, v in cv_bal.items()},
}, "etapa4_parametros.json")

print("\n" + "=" * 70 + "\n" + md + "\nArquivos em:", OUT_DIR)